# Objective 2

## Read-Only Search-Space Profiler

## Metric Definitions

**Hop indexing.** Hop `h` executes the planned relation `target_rule[h]`
(0-indexed), corresponding to $r_{h+1}$ in the paper notation.
For a relation plan of length $L$, the final hop is $h=L-1$.

| Counter / Metric | Definition |
|---|---|
| `active_prefixes[h]` | Number of active path-prefix states $|\mathcal{P}_h|$ presented for expansion at hop `h`. Counted before the node-in-graph check; therefore, a start entity absent from its local graph still occupies the initial frontier. |
| `unique_expanded_nodes[h]` | Number of distinct entities whose adjacency lists are actually inspected at hop `h`. Multiple path prefixes ending at the same entity contribute only once to this unique-entity count. |
| `edges_examined[h]` | Number of adjacency entries inspected at hop `h` before relation matching. This is counted per path-prefix expansion; if the same entity is reached through multiple prefixes and expanded repeatedly, each adjacency inspection is counted. |
| `candidate_branches[h]` | Number of relation-valid candidate extensions generated at hop `h`, i.e. $|\mathcal{C}_h|$. These are the branches satisfying the required relation $r_{h+1}$. This has the same counting semantics as the legacy `hop_frontier` counter in Objective 1. |
| `branch_expansion_ratio[h]` | Relation-valid branch expansion ratio $\rho_h = |\mathcal{C}_h| / |\mathcal{P}_h|$, defined only when $|\mathcal{P}_h| > 0$. Values $>1$, $=1$, and $<1$ indicate frontier expansion, unchanged size, and contraction, respectively. |
| `unique_frontier_nodes[h]` | Number of distinct endpoint entities in the relation-valid candidate frontier generated at hop `h`, corresponding to $|\mathcal{U}_{h+1}|$. Distinct path prefixes may terminate at the same entity. |
| `retrieved_paths` | Number of complete reasoning paths produced after execution of the current topic-entity/relation-plan traversal. |
| `peak_frontier_size` | Maximum number of active path prefixes over the hops of a traversal, i.e. $\max_h |\mathcal{P}_h|$. At question level, Peak Frontier is the maximum across all executed topic entities, relation plans, and hops. |
| `peak_queue_size` | Maximum physical `deque` length observed during implementation-level BFS execution. This is retained only as a diagnostic and is **not** treated as the paper's Peak Frontier metric. |
| `afp_eligible` | Indicates whether the relation plan contains at least one intermediate pruning position. Under final-hop protection, a plan is AFP-eligible when $L \geq 2$. |
| `decision_opportunity[h]` | Indicates an intermediate hop at which AFP would have an actual branch-selection decision: $h < L-1$ and $|\mathcal{C}_h| > 1$. |
| `downstream_expansion_edges[h]` | Remaining baseline edge-examination work after hop `h`, defined as $D_h = \sum_{t=h+1}^{L-1} E_t$. This measures the amount of future graph expansion potentially affected by a pruning decision made at hop `h`. |

**Aggregation rule.** Path-prefix, edge, and candidate-branch counts are
aggregated across topic entities and relation plans by **summing** their
counts. Quantities representing distinct entities are aggregated by **set
union**, never by summing per-call unique counts. Peak Frontier is obtained
by a **maximum**, not a sum.

**Interpretation.** The counters are intentionally complementary.
$|\mathcal{P}_h|$ and $|\mathcal{C}_h|$ characterize path-level branching,
unique-entity counters characterize entity-level exploration, and
`edges_examined[h]` measures the actual adjacency work performed by
relation-constrained retrieval. The expansion ratio $\rho_h$ shows how the
relation-valid frontier changes between hops, while
`downstream_expansion_edges[h]` indicates how much future traversal remains
after a possible intermediate pruning decision.

In [69]:
# ## 2.2 Profiled BFS v2 — READ-ONLY per-hop search-space profiler
# Traversal below is copied VERBATIM from bfs_with_rule_instrumented (cell 23), which is
# itself copied verbatim from src/utils/graph_utils.py. Only observation counters were
# added. Returns (result_paths, profile): result_paths must be identical to official
# bfs_with_rule output in contents, order, AND multiplicity.

from collections import deque
import time

def bfs_with_rule_profiled_v2(graph, start_node, target_rule):
    t0 = time.perf_counter()
    L = len(target_rule)
    result_paths = []
    active_prefixes = [0] * L                     # |P_h| dequeued for expansion
    unique_expanded_sets = [set() for _ in range(L)]
    edges_examined = [0] * L                      # adjacency entries inspected pre-match
    candidate_branches = [0] * L                  # == legacy hop_frontier semantics
    unique_frontier_sets = [set() for _ in range(L)]
    peak_queue_size = 1                           # seed state occupies the queue initially
    queue = deque([(start_node, [])])
    while queue:
        if len(queue) > peak_queue_size:
            peak_queue_size = len(queue)
        current_node, current_path = queue.popleft()
        if len(current_path) == L:
            result_paths.append(current_path)
        if len(current_path) < L:
            active_prefixes[len(current_path)] += 1
            if current_node not in graph:
                continue
            hop_idx = len(current_path)
            unique_expanded_sets[hop_idx].add(current_node)
            for neighbor in graph.neighbors(current_node):
                edges_examined[hop_idx] += 1
                rel = graph[current_node][neighbor]["relation"]
                if rel != target_rule[hop_idx] or len(current_path) > len(target_rule):
                    continue
                queue.append((neighbor, current_path + [(current_node, rel, neighbor)]))
                candidate_branches[hop_idx] += 1
                unique_frontier_sets[hop_idx].add(neighbor)
    profile = {
        "plan_length": L,
        "active_prefixes": active_prefixes,
        "unique_expanded_nodes": [len(s) for s in unique_expanded_sets],
        "edges_examined": edges_examined,
        "candidate_branches": candidate_branches,
        "unique_frontier_nodes": [len(s) for s in unique_frontier_sets],
        "peak_active_prefixes": max(active_prefixes) if L > 0 else 0,
        "peak_queue_size": peak_queue_size,
        "retrieved_paths": len(result_paths),
        "time_sec_perf": time.perf_counter() - t0,
        "_unique_expanded_node_sets": unique_expanded_sets,   # raw sets, for correct union-aggregation
        "_unique_frontier_node_sets": unique_frontier_sets,   # raw sets, for correct union-aggregation
    }
    return result_paths, profile

print("bfs_with_rule_profiled_v2 defined.")

bfs_with_rule_profiled_v2 defined.


## Official-vs-profiled equivalence test (hard gate)

In [70]:

# ## 2.3 Official-vs-profiled equivalence test (WebQSP + CWQ, ALL frozen test calls)
# Hard gate per PRD §13: bfs_with_rule_profiled_v2 must reproduce official bfs_with_rule
# EXACTLY -- same paths, order, multiplicity -- before any profile data is trusted.

from datasets import load_dataset
from tqdm.auto import tqdm

# Build graph lookup maps from dataset objects (since in-memory records had 'graph' popped to save RAM)
if "full_test_all" not in globals():
    full_test_all = load_dataset(DATASET_NAME, split=SPLIT)
webqsp_graph_map = {sample["id"]: sample["graph"] for sample in full_test_all}

if "cwq_full_test_all" not in globals():
    cwq_full_test_all = load_dataset(CWQ_DATASET_NAME, split=CWQ_SPLIT)
cwq_graph_map = {sample["id"]: sample["graph"] for sample in cwq_full_test_all}

def run_equivalence_test(planning_recs, graph_map, dataset_label):
    checked = mismatches = reachability_mismatches = path_count_mismatches = 0
    for rec in tqdm(planning_recs, desc=f"Equivalence ({dataset_label})"):
        # Retrieve graph from rec if present, otherwise lookup from dataset map
        graph_data = rec.get("graph") or graph_map[rec["id"]]
        graph = build_graph(graph_data)
        gold_answers = set(rec["a_entity"])

        for entity in rec["q_entity"]:
            for rule in rec["predicted_paths"]:
                if len(rule) == 0:
                    continue  # Baseline retrieval loops skip empty relation plans

                checked += 1
                official_result = bfs_with_rule(graph, entity, rule)
                profiled_result, _ = bfs_with_rule_profiled_v2(graph, entity, rule)

                # Check 1: Exact path list equality (order + items)
                if official_result != profiled_result:
                    mismatches += 1
                    continue

                # Check 2: Path count equality
                if len(official_result) != len(profiled_result):
                    path_count_mismatches += 1

                # Check 3: Gold answer reachability match
                official_reachable = len(path_endpoints(official_result) & gold_answers) > 0
                profiled_reachable = len(path_endpoints(profiled_result) & gold_answers) > 0
                if official_reachable != profiled_reachable:
                    reachability_mismatches += 1

    print(f"\n[{dataset_label}] Checked {checked} calls | mismatches: {mismatches} | "
              f"reachability mismatches: {reachability_mismatches} | path-count mismatches: {path_count_mismatches}")
    return checked, mismatches, reachability_mismatches, path_count_mismatches

# 1. Run WebQSP Hard Gate
webqsp_checked, webqsp_mm, webqsp_reach_mm, webqsp_count_mm = run_equivalence_test(
    planning_records_full, webqsp_graph_map, "WebQSP test"
)
assert webqsp_mm == 0 and webqsp_reach_mm == 0 and webqsp_count_mm == 0, "Diverges on WebQSP -- STOP."

# 2. Run CWQ Hard Gate
cwq_checked, cwq_mm, cwq_reach_mm, cwq_count_mm = run_equivalence_test(
    cwq_planning_records_full, cwq_graph_map, "CWQ test"
)
assert cwq_mm == 0 and cwq_reach_mm == 0 and cwq_count_mm == 0, "Diverges on CWQ -- STOP."

print("\n=== EQUIVALENCE GATE: PASSED ===")
print(f"WebQSP: {webqsp_checked} calls, 0 mismatches")
print(f"CWQ:    {cwq_checked} calls, 0 mismatches")

Equivalence (WebQSP test):   0%|          | 0/1628 [00:00<?, ?it/s]


[WebQSP test] Checked 4937 calls | mismatches: 0 | reachability mismatches: 0 | path-count mismatches: 0


Equivalence (CWQ test):   0%|          | 0/3531 [00:00<?, ?it/s]


[CWQ test] Checked 16421 calls | mismatches: 0 | reachability mismatches: 0 | path-count mismatches: 0

=== EQUIVALENCE GATE: PASSED ===
WebQSP: 4937 calls, 0 mismatches
CWQ:    16421 calls, 0 mismatches


## Freeze validation relation plans (dev inputs only, test gold untouched)

In [71]:
# ## 2.4 Freeze validation relation plans (WebQSP + CWQ)
# Dev-only planner runs, checkpointed like the frozen test runs. Never touches test gold.
RQ1_DEV_DIR = "/kaggle/working/step2_rq1_dev"
os.makedirs(RQ1_DEV_DIR, exist_ok=True)

WEBQSP_VAL_PLANNING_CKPT = os.path.join(RQ1_DEV_DIR, "planning_webqsp_validation.jsonl")
CWQ_VAL_PLANNING_CKPT = os.path.join(RQ1_DEV_DIR, "planning_cwq_validation.jsonl")

SEED = 42
torch.manual_seed(SEED)

webqsp_val = load_dataset(DATASET_NAME, split="validation")
cwq_val = load_dataset(CWQ_DATASET_NAME, split="validation")
print(f"WebQSP validation: {len(webqsp_val)} questions")
print(f"CWQ validation: {len(cwq_val)} questions  (open item: not yet in verified-facts.md -- record this number there)")

def run_planning_checkpointed(dataset_split, ckpt_path, desc):
    done = load_checkpoint(ckpt_path)
    print(f"Resuming: {len(done)} / {len(dataset_split)} already planned ({desc}).")
    with open(ckpt_path, "a") as fout:
        for sample in tqdm(dataset_split, desc=desc):
            if sample["id"] in done:
                continue
            input_text = prompter.format(instruction=INSTRUCTION, message=sample["question"])
            t0 = time.time()
            raw_output = generate_seq(model, input_text, tokenizer, num_beam=N_BEAM, do_sample=True, max_new_tokens=100)
            planning_time = time.time() - t0
            rel_paths = parse_prediction(raw_output["paths"])
            rec = {
                "id": sample["id"], "question": sample["question"],
                "q_entity": sample["q_entity"], "a_entity": sample["a_entity"],
                "graph": sample["graph"], "predicted_paths": rel_paths,
                "planning_time_sec": planning_time,
            }
            fout.write(json.dumps(rec) + "\n")
            fout.flush()
            done[sample["id"]] = rec
    return list(done.values())

webqsp_val_planning = run_planning_checkpointed(webqsp_val, WEBQSP_VAL_PLANNING_CKPT, "Planning (WebQSP validation)")
cwq_val_planning = run_planning_checkpointed(cwq_val, CWQ_VAL_PLANNING_CKPT, "Planning (CWQ validation)")

print(f"WebQSP validation plans frozen: {len(webqsp_val_planning)}")
print(f"CWQ validation plans frozen: {len(cwq_val_planning)}")

WebQSP validation: 246 questions
CWQ validation: 3519 questions  (open item: not yet in verified-facts.md -- record this number there)
Resuming: 246 / 246 already planned (Planning (WebQSP validation)).


Planning (WebQSP validation):   0%|          | 0/246 [00:00<?, ?it/s]

Resuming: 3519 / 3519 already planned (Planning (CWQ validation)).


Planning (CWQ validation):   0%|          | 0/3519 [00:00<?, ?it/s]

WebQSP validation plans frozen: 246
CWQ validation plans frozen: 3519


## Development RQ1 profiling over frozen validation plans

In [72]:
# ## 2.5 RQ1 profiling over frozen validation plans -- raw question x plan x hop logs
# Schema per PRD §11 & Paper Definitions:
# rho_h, decision opportunities, downstream edges, peak frontier.
#
# IMPORTANT:
# - Prefix/edge/branch counts are SUMMED across topic entities.
# - Unique-node quantities are aggregated by SET UNION across topic entities.
# - Decision opportunity is detected within an actual topic-entity traversal.
# - Peak Frontier is the maximum active-prefix frontier observed in any traversal.
# - peak_queue_size is retained only as an implementation diagnostic.

import os
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

def profile_validation_set(planning_recs, dataset_label):
    rows = []

    for rec in tqdm(planning_recs, desc=f"RQ1 profiling ({dataset_label})"):
        graph = build_graph(rec["graph"])
        gold_answers = set(rec["a_entity"])
        topic_entity_count = len(rec["q_entity"])

        # First pass: determine plan-level and question-level answer reachability
        plan_reachable_flags = []
        for rule in rec["predicted_paths"]:
            plan_paths = []
            if len(rule) > 0:
                for entity in rec["q_entity"]:
                    paths, _ = bfs_with_rule_profiled_v2(graph, entity, rule)
                    plan_paths.extend(paths)

            plan_reachable = len(path_endpoints(plan_paths) & gold_answers) > 0
            plan_reachable_flags.append(plan_reachable)

        question_gold_reachable = any(plan_reachable_flags)

        # Second pass: collect raw RQ1 profiling statistics
        for plan_rank, rule in enumerate(rec["predicted_paths"]):
            L = len(rule)
            if L == 0:
                continue

            t0 = time.perf_counter()

            agg_active = [0] * L
            agg_edges = [0] * L
            agg_branches = [0] * L
            union_expanded = [set() for _ in range(L)]
            union_frontier = [set() for _ in range(L)]

            total_retrieved_paths = 0
            peak_queue = 0
            plan_peak_frontier = 0
            decision_flags = [False] * L

            for entity in rec["q_entity"]:
                _, profile = bfs_with_rule_profiled_v2(graph, entity, rule)

                entity_peak_frontier = (
                    max(profile["active_prefixes"])
                    if len(profile["active_prefixes"]) > 0 else 0
                )
                plan_peak_frontier = max(plan_peak_frontier, entity_peak_frontier)
                peak_queue = max(peak_queue, profile["peak_queue_size"])

                for h in range(L):
                    agg_active[h] += profile["active_prefixes"][h]
                    agg_edges[h] += profile["edges_examined"][h]
                    agg_branches[h] += profile["candidate_branches"][h]
                    union_expanded[h] |= profile["_unique_expanded_node_sets"][h]
                    union_frontier[h] |= profile["_unique_frontier_node_sets"][h]

                    if h < L - 1 and profile["candidate_branches"][h] > 1:
                        decision_flags[h] = True

                total_retrieved_paths += profile["retrieved_paths"]

            retrieval_time = time.perf_counter() - t0
            plan_gold_reachable = plan_reachable_flags[plan_rank]
            plan_afp_eligible = (L >= 2)

            for h in range(L):
                # rho_h = |C_h| / |P_h|
                branch_expansion_ratio = (
                    agg_branches[h] / agg_active[h]
                    if agg_active[h] > 0 else np.nan
                )

                # AFP decision opportunity at an intermediate hop
                decision_opportunity = decision_flags[h]

                # D_h = sum of future baseline edge examinations after hop h
                downstream_expansion_edges = (
                    sum(agg_edges[h + 1:]) if h < L - 1 else 0
                )

                rows.append({
                    "dataset": dataset_label,
                    "question_id": rec["id"],
                    "plan_id": f"{rec['id']}_p{plan_rank}",
                    "plan_rank": plan_rank,
                    "plan_length": L,
                    "hop": h,
                    "required_relation": rule[h],
                    "topic_entity_count": topic_entity_count,

                    "active_prefixes": agg_active[h],
                    "unique_expanded_nodes": len(union_expanded[h]),
                    "edges_examined": agg_edges[h],
                    "candidate_branches": agg_branches[h],
                    "unique_frontier_nodes": len(union_frontier[h]),
                    "branch_expansion_ratio": branch_expansion_ratio,

                    "retrieved_paths_final_plan": total_retrieved_paths,
                    "peak_frontier_size_plan": plan_peak_frontier,
                    "peak_queue_size_plan": peak_queue,

                    "plan_gold_reachable": plan_gold_reachable,
                    "question_gold_reachable": question_gold_reachable,

                    "afp_eligible_plan": plan_afp_eligible,
                    "decision_opportunity": decision_opportunity,
                    "downstream_expansion_edges": downstream_expansion_edges,

                    "retrieval_time_sec_plan": retrieval_time,
                })

    return pd.DataFrame(rows)


# =========================================================
# Run development RQ1 profiling
# =========================================================
df_rq1_webqsp_dev = profile_validation_set(webqsp_val_planning, "webqsp")
df_rq1_cwq_dev = profile_validation_set(cwq_val_planning, "cwq")


# =========================================================
# Save raw question x plan x hop logs
# =========================================================
RQ1_DEV_DIR_OUT = RQ1_DEV_DIR

df_rq1_webqsp_dev.to_csv(
    os.path.join(RQ1_DEV_DIR_OUT, "rq1_webqsp_dev_plan_hop.csv"),
    index=False
)
df_rq1_cwq_dev.to_csv(
    os.path.join(RQ1_DEV_DIR_OUT, "rq1_cwq_dev_plan_hop.csv"),
    index=False
)
df_rq1_webqsp_dev.to_json(
    os.path.join(RQ1_DEV_DIR_OUT, "rq1_webqsp_dev_plan_hop.jsonl"),
    orient="records",
    lines=True
)
df_rq1_cwq_dev.to_json(
    os.path.join(RQ1_DEV_DIR_OUT, "rq1_cwq_dev_plan_hop.jsonl"),
    orient="records",
    lines=True
)

print(f"WebQSP dev plan-hop rows: {len(df_rq1_webqsp_dev)}")
print(f"CWQ dev plan-hop rows: {len(df_rq1_cwq_dev)}")

RQ1 profiling (webqsp):   0%|          | 0/246 [00:00<?, ?it/s]

RQ1 profiling (cwq):   0%|          | 0/3519 [00:00<?, ?it/s]

WebQSP dev plan-hop rows: 1034
CWQ dev plan-hop rows: 18851


## Quick Sanity Check

In [73]:
# ## 2.5.1 Sanity checks for raw RQ1 profiling logs

def sanity_check_rq1(df, label):
    print(f"\n=== {label} RQ1 RAW PROFILE SANITY CHECK ===")
    print(f"Rows: {len(df)}")
    print(f"Questions: {df['question_id'].nunique()}")
    print(f"Plans: {df['plan_id'].nunique()}")
    print(f"Hop range: {df['hop'].min()} -> {df['hop'].max()}")

    # Basic non-negativity
    count_cols = [
        "active_prefixes",
        "unique_expanded_nodes",
        "edges_examined",
        "candidate_branches",
        "unique_frontier_nodes",
        "retrieved_paths_final_plan",
        "peak_frontier_size_plan",
        "peak_queue_size_plan",
        "downstream_expansion_edges",
    ]

    for col in count_cols:
        print(f"{col}: negative values = {(df[col] < 0).sum()}")

    # rho_h should be NaN only when active_prefixes == 0
    invalid_rho_nan = df[
        df["branch_expansion_ratio"].isna()
        & (df["active_prefixes"] > 0)
    ]

    invalid_rho_defined = df[
        df["branch_expansion_ratio"].notna()
        & (df["active_prefixes"] == 0)
    ]

    print(
        "rho NaN despite active_prefixes > 0:",
        len(invalid_rho_nan)
    )
    print(
        "rho defined despite active_prefixes == 0:",
        len(invalid_rho_defined)
    )

    # Candidate branches should be zero if no active prefixes exist
    print(
        "candidate_branches > 0 with active_prefixes == 0:",
        len(df[
            (df["active_prefixes"] == 0)
            & (df["candidate_branches"] > 0)
        ])
    )

    # Decision opportunities must never occur on final hop
    print(
        "decision opportunity on final hop:",
        len(df[
            df["decision_opportunity"]
            & (df["hop"] == df["plan_length"] - 1)
        ])
    )

    # Decision opportunities must belong to AFP-eligible plans
    print(
        "decision opportunity on non-AFP-eligible plan:",
        len(df[
            df["decision_opportunity"]
            & (~df["afp_eligible_plan"])
        ])
    )

    # Final hop must have zero downstream expansion
    print(
        "nonzero downstream edges at final hop:",
        len(df[
            (df["hop"] == df["plan_length"] - 1)
            & (df["downstream_expansion_edges"] != 0)
        ])
    )

    # One row per question x plan x hop
    duplicates = df.duplicated(
        subset=["question_id", "plan_id", "hop"]
    ).sum()

    print(
        "duplicate question-plan-hop rows:",
        duplicates
    )


sanity_check_rq1(df_rq1_webqsp_dev, "WebQSP")
sanity_check_rq1(df_rq1_cwq_dev, "CWQ")


=== WebQSP RQ1 RAW PROFILE SANITY CHECK ===
Rows: 1034
Questions: 246
Plans: 721
Hop range: 0 -> 1
active_prefixes: negative values = 0
unique_expanded_nodes: negative values = 0
edges_examined: negative values = 0
candidate_branches: negative values = 0
unique_frontier_nodes: negative values = 0
retrieved_paths_final_plan: negative values = 0
peak_frontier_size_plan: negative values = 0
peak_queue_size_plan: negative values = 0
downstream_expansion_edges: negative values = 0
rho NaN despite active_prefixes > 0: 0
rho defined despite active_prefixes == 0: 0
candidate_branches > 0 with active_prefixes == 0: 0
decision opportunity on final hop: 0
decision opportunity on non-AFP-eligible plan: 0
nonzero downstream edges at final hop: 0
duplicate question-plan-hop rows: 0

=== CWQ RQ1 RAW PROFILE SANITY CHECK ===
Rows: 18851
Questions: 3519
Plans: 10529
Hop range: 0 -> 4
active_prefixes: negative values = 0
unique_expanded_nodes: negative values = 0
edges_examined: negative values = 0
can

## Per-hop descriptive statistics and branching prevalence

In [74]:
# ## 2.6 RQ1 development descriptive analysis
# Uses ONLY frozen validation profiling outputs from Cell 141.
# Produces hop-level, plan-level, and question-level summaries required for RQ1.
#
# IMPORTANT:
# - "all_active" includes only observed frontiers with active_prefixes > 0.
# - "intermediate_active" excludes final hops and is most relevant to AFP.
# - Branching prevalence uses aggregated question x plan x hop candidate counts.
# - decision_opportunity is stricter: >1 candidate within an actual topic-entity traversal.
# - Downstream D_h is NOT summed as "avoidable work" because different decision hops can overlap.
# - Unique-node counts are never summed into fake question-level unique counts.

import os
import numpy as np
import pandas as pd

PRIMARY_VARS = [
    "active_prefixes",
    "unique_expanded_nodes",
    "edges_examined",
    "candidate_branches",
    "unique_frontier_nodes",
]

PCTS = [0.50, 0.75, 0.90, 0.95]
BRANCH_THRESHOLDS = [1, 5, 10, 50]

def series_summary(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) == 0:
        return {
            "n": 0, "mean": np.nan, "median": np.nan,
            "p75": np.nan, "p90": np.nan,
            "p95": np.nan, "max": np.nan
        }
    return {
        "n": len(s),
        "mean": s.mean(),
        "median": s.median(),
        "p75": s.quantile(0.75),
        "p90": s.quantile(0.90),
        "p95": s.quantile(0.95),
        "max": s.max(),
    }

def distribution_table(df, label):
    rows = []
    scopes = {
        "all_active": df[df["active_prefixes"] > 0],
        "intermediate_active": df[
            (df["active_prefixes"] > 0) &
            (df["hop"] < df["plan_length"] - 1)
        ],
    }

    for scope_name, scope_df in scopes.items():
        groups = [("ALL", scope_df)]
        groups += [(str(h), g) for h, g in scope_df.groupby("hop")]

        for hop_label, g in groups:
            for metric in PRIMARY_VARS:
                s = series_summary(g[metric])
                rows.append({
                    "dataset": label,
                    "scope": scope_name,
                    "hop": hop_label,
                    "metric": metric,
                    **s
                })

    return pd.DataFrame(rows)

def rho_table(df, label):
    rows = []
    scopes = {
        "all_active": df[df["active_prefixes"] > 0],
        "intermediate_active": df[
            (df["active_prefixes"] > 0) &
            (df["hop"] < df["plan_length"] - 1)
        ],
    }

    for scope_name, scope_df in scopes.items():
        groups = [("ALL", scope_df)]
        groups += [(str(h), g) for h, g in scope_df.groupby("hop")]

        for hop_label, g in groups:
            r = g["branch_expansion_ratio"].dropna()
            s = series_summary(r)

            rows.append({
                "dataset": label,
                "scope": scope_name,
                "hop": hop_label,
                **s,
                "rho_gt_1_pct": 100 * (r > 1).mean() if len(r) else np.nan,
                "rho_eq_1_pct": 100 * np.isclose(r, 1.0).mean() if len(r) else np.nan,
                "rho_lt_1_pct": 100 * (r < 1).mean() if len(r) else np.nan,
            })

    return pd.DataFrame(rows)

def branching_prevalence_table(df, label):
    rows = []
    scopes = {
        "all_active": df[df["active_prefixes"] > 0],
        "intermediate_active": df[
            (df["active_prefixes"] > 0) &
            (df["hop"] < df["plan_length"] - 1)
        ],
    }

    for scope_name, scope_df in scopes.items():
        groups = [("ALL", scope_df)]
        groups += [(str(h), g) for h, g in scope_df.groupby("hop")]

        for hop_label, g in groups:
            row = {
                "dataset": label,
                "scope": scope_name,
                "hop": hop_label,
                "n_frontiers": len(g),
            }

            for t in BRANCH_THRESHOLDS:
                row[f"candidate_gt_{t}_pct"] = (
                    100 * (g["candidate_branches"] > t).mean()
                    if len(g) else np.nan
                )

            rows.append(row)

    return pd.DataFrame(rows)

def duplicate_pressure_table(df, label):
    work = df[df["active_prefixes"] > 0].copy()

    # Paper ratio |P_h| / |X_h|.
    # Undefined when no entity adjacency was actually expanded.
    work["prefix_to_unique_expanded_ratio"] = np.where(
        work["unique_expanded_nodes"] > 0,
        work["active_prefixes"] / work["unique_expanded_nodes"],
        np.nan
    )

    # Paper ratio |C_h| / |U_{h+1}|.
    # This directly measures repeated candidate endpoints.
    work["candidate_to_unique_frontier_ratio"] = np.where(
        work["unique_frontier_nodes"] > 0,
        work["candidate_branches"] / work["unique_frontier_nodes"],
        np.nan
    )

    rows = []
    metrics = [
        "prefix_to_unique_expanded_ratio",
        "candidate_to_unique_frontier_ratio",
    ]

    groups = [("ALL", work)]
    groups += [(str(h), g) for h, g in work.groupby("hop")]

    for hop_label, g in groups:
        for metric in metrics:
            s = series_summary(g[metric])

            rows.append({
                "dataset": label,
                "hop": hop_label,
                "metric": metric,
                **s,
                "ratio_gt_1_pct": (
                    100 * (g[metric].dropna() > 1).mean()
                    if g[metric].notna().any() else np.nan
                )
            })

    return pd.DataFrame(rows)

def plan_question_tables(df, label):
    plan_df = (
        df.groupby(
            ["dataset", "question_id", "plan_id", "plan_rank"],
            as_index=False
        )
        .agg(
            plan_length=("plan_length", "first"),
            retrieved_paths_final_plan=("retrieved_paths_final_plan", "first"),
            peak_frontier_size_plan=("peak_frontier_size_plan", "first"),
            peak_queue_size_plan=("peak_queue_size_plan", "first"),
            afp_eligible_plan=("afp_eligible_plan", "first"),
            plan_gold_reachable=("plan_gold_reachable", "first"),
            question_gold_reachable=("question_gold_reachable", "first"),
            plan_has_decision=("decision_opportunity", "max"),
            retrieval_time_sec_plan=("retrieval_time_sec_plan", "first"),
        )
    )

    question_work = (
        df.groupby("question_id", as_index=False)
        .agg(
            active_prefixes_total=("active_prefixes", "sum"),
            edges_examined_total=("edges_examined", "sum"),
            candidate_branches_total=("candidate_branches", "sum"),
        )
    )

    question_plan = (
        plan_df.groupby("question_id", as_index=False)
        .agg(
            n_plans=("plan_id", "nunique"),
            retrieved_paths_total=("retrieved_paths_final_plan", "sum"),
            peak_frontier_size_question=("peak_frontier_size_plan", "max"),
            afp_eligible_question=("afp_eligible_plan", "max"),
            decision_opportunity_question=("plan_has_decision", "max"),
            question_gold_reachable=("question_gold_reachable", "max"),
        )
    )

    question_df = question_work.merge(
        question_plan,
        on="question_id",
        how="inner"
    )

    plan_rows = []
    for metric in [
        "retrieved_paths_final_plan",
        "peak_frontier_size_plan",
    ]:
        plan_rows.append({
            "dataset": label,
            "metric": metric,
            **series_summary(plan_df[metric])
        })

    question_rows = []
    for metric in [
        "active_prefixes_total",
        "edges_examined_total",
        "candidate_branches_total",
        "retrieved_paths_total",
        "peak_frontier_size_question",
    ]:
        question_rows.append({
            "dataset": label,
            "metric": metric,
            **series_summary(question_df[metric])
        })

    return (
        plan_df,
        question_df,
        pd.DataFrame(plan_rows),
        pd.DataFrame(question_rows)
    )

def opportunity_table(df, plan_df, question_df, label):
    intermediate_active = df[
        (df["active_prefixes"] > 0) &
        (df["hop"] < df["plan_length"] - 1)
    ]

    decision_hops = intermediate_active[
        intermediate_active["decision_opportunity"]
    ]

    eligible_plans = plan_df[plan_df["afp_eligible_plan"]]
    eligible_questions = question_df[
        question_df["afp_eligible_question"]
    ]

    rows = [{
        "dataset": label,
        "questions": question_df["question_id"].nunique(),
        "plans": len(plan_df),
        "afp_eligible_plans": int(plan_df["afp_eligible_plan"].sum()),
        "afp_eligible_plans_pct": 100 * plan_df["afp_eligible_plan"].mean(),
        "plans_with_decision": int(plan_df["plan_has_decision"].sum()),
        "plans_with_decision_pct": 100 * plan_df["plan_has_decision"].mean(),
        "decision_pct_among_eligible_plans": (
            100 * eligible_plans["plan_has_decision"].mean()
            if len(eligible_plans) else np.nan
        ),
        "afp_eligible_questions": int(question_df["afp_eligible_question"].sum()),
        "afp_eligible_questions_pct": 100 * question_df["afp_eligible_question"].mean(),
        "questions_with_decision": int(question_df["decision_opportunity_question"].sum()),
        "questions_with_decision_pct": 100 * question_df["decision_opportunity_question"].mean(),
        "decision_pct_among_eligible_questions": (
            100 * eligible_questions["decision_opportunity_question"].mean()
            if len(eligible_questions) else np.nan
        ),
        "intermediate_active_hops": len(intermediate_active),
        "decision_hops": len(decision_hops),
        "decision_hop_pct": (
            100 * len(decision_hops) / len(intermediate_active)
            if len(intermediate_active) else np.nan
        ),
        "decision_hops_with_future_work": int(
            (decision_hops["downstream_expansion_edges"] > 0).sum()
        ),
        "decision_hops_with_future_work_pct": (
            100 * (decision_hops["downstream_expansion_edges"] > 0).mean()
            if len(decision_hops) else np.nan
        ),
    }]

    return pd.DataFrame(rows)

def downstream_table(df, label):
    decision_df = df[
        df["decision_opportunity"] &
        (df["hop"] < df["plan_length"] - 1)
    ].copy()

    rows = []
    groups = [("ALL", decision_df)]
    groups += [(str(h), g) for h, g in decision_df.groupby("hop")]

    for hop_label, g in groups:
        s = series_summary(g["downstream_expansion_edges"])

        rows.append({
            "dataset": label,
            "hop": hop_label,
            **s,
            "future_work_gt_0_pct": (
                100 * (g["downstream_expansion_edges"] > 0).mean()
                if len(g) else np.nan
            )
        })

    return pd.DataFrame(rows)

def reachability_table(plan_df, question_df, label):
    return pd.DataFrame([{
        "dataset": label,
        "plans": len(plan_df),
        "reachable_plans": int(plan_df["plan_gold_reachable"].sum()),
        "plan_reachability_pct": 100 * plan_df["plan_gold_reachable"].mean(),
        "questions": len(question_df),
        "reachable_questions": int(question_df["question_gold_reachable"].sum()),
        "question_reachability_pct": 100 * question_df["question_gold_reachable"].mean(),
    }])

def analyze_rq1_dev(df, label):
    dist = distribution_table(df, label)
    rho = rho_table(df, label)
    branching = branching_prevalence_table(df, label)
    duplicate = duplicate_pressure_table(df, label)

    plan_df, question_df, plan_summary, question_summary = (
        plan_question_tables(df, label)
    )

    opportunity = opportunity_table(
        df, plan_df, question_df, label
    )

    downstream = downstream_table(df, label)
    reachability = reachability_table(
        plan_df, question_df, label
    )

    prefix_undefined = int(
        (
            (df["active_prefixes"] > 0) &
            (df["unique_expanded_nodes"] == 0)
        ).sum()
    )

    candidate_undefined = int(
        (
            (df["candidate_branches"] > 0) &
            (df["unique_frontier_nodes"] == 0)
        ).sum()
    )

    print(f"\n{'='*80}")
    print(f"{label.upper()} -- DEVELOPMENT RQ1 CHARACTERIZATION")
    print(f"{'='*80}")

    print("\n[Primary search-space distributions: all active frontiers]")
    display(
        dist[
            (dist["scope"] == "all_active") &
            (dist["hop"] == "ALL")
        ].reset_index(drop=True)
    )

    print("\n[Primary search-space distributions: intermediate active frontiers]")
    display(
        dist[
            (dist["scope"] == "intermediate_active") &
            (dist["hop"] == "ALL")
        ].reset_index(drop=True)
    )

    print("\n[Hop-wise rho_h]")
    display(
        rho[
            rho["scope"] == "all_active"
        ].reset_index(drop=True)
    )

    print("\n[Branching prevalence]")
    display(
        branching[
            branching["hop"] == "ALL"
        ].reset_index(drop=True)
    )

    print("\n[AFP eligibility and actual decision opportunities]")
    display(opportunity)

    print("\n[Downstream expansion at actual decision hops]")
    display(downstream)

    print("\n[Duplicate search pressure]")
    display(
        duplicate[
            duplicate["hop"] == "ALL"
        ].reset_index(drop=True)
    )

    print(
        f"Prefix-duplicate ratio undefined because unique_expanded_nodes=0: "
        f"{prefix_undefined} rows"
    )
    print(
        f"Candidate-duplicate ratio undefined despite candidate_branches>0: "
        f"{candidate_undefined} rows"
    )

    print("\n[Plan-level distributions]")
    display(plan_summary)

    print("\n[Question-level work distributions]")
    display(question_summary)

    print("\n[Reachability]")
    display(reachability)

    return {
        "distribution": dist,
        "rho": rho,
        "branching": branching,
        "duplicate": duplicate,
        "plan_raw": plan_df,
        "question_raw": question_df,
        "plan_summary": plan_summary,
        "question_summary": question_summary,
        "opportunity": opportunity,
        "downstream": downstream,
        "reachability": reachability,
    }


# =========================================================
# Run RQ1 development descriptive analysis
# =========================================================
rq1_webqsp_summary = analyze_rq1_dev(
    df_rq1_webqsp_dev,
    "webqsp"
)

rq1_cwq_summary = analyze_rq1_dev(
    df_rq1_cwq_dev,
    "cwq"
)


# =========================================================
# Save aggregated development summaries separately
# =========================================================
def save_rq1_summaries(summary_dict, dataset_label):
    for name, table in summary_dict.items():
        table.to_csv(
            os.path.join(
                RQ1_DEV_DIR,
                f"rq1_{dataset_label}_dev_{name}.csv"
            ),
            index=False
        )

save_rq1_summaries(rq1_webqsp_summary, "webqsp")
save_rq1_summaries(rq1_cwq_summary, "cwq")

print("\nRQ1 development descriptive summaries saved.")


WEBQSP -- DEVELOPMENT RQ1 CHARACTERIZATION

[Primary search-space distributions: all active frontiers]


,dataset,scope,hop,metric,n,mean,median,p75,p90,p95,max
0,webqsp,all_active,ALL,active_prefixes,971,2.512873,1.0,1.0,4.0,9.5,56
1,webqsp,all_active,ALL,unique_expanded_nodes,971,2.512873,1.0,1.0,4.0,9.5,56
2,webqsp,all_active,ALL,edges_examined,971,351.726056,135.0,402.0,1475.0,1686.0,1969
3,webqsp,all_active,ALL,candidate_branches,971,8.221421,1.0,5.0,18.0,35.0,533
4,webqsp,all_active,ALL,unique_frontier_nodes,971,6.423275,1.0,5.0,17.0,33.0,145



[Primary search-space distributions: intermediate active frontiers]


,dataset,scope,hop,metric,n,mean,median,p75,p90,p95,max
0,webqsp,intermediate_active,ALL,active_prefixes,312,1.048077,1.0,1.00,1.0,1.0,2
1,webqsp,intermediate_active,ALL,unique_expanded_nodes,312,1.048077,1.0,1.00,1.0,1.0,2
2,webqsp,intermediate_active,ALL,edges_examined,312,316.525641,161.0,354.75,569.2,1641.0,1915
3,webqsp,intermediate_active,ALL,candidate_branches,312,5.471154,1.0,5.00,16.0,27.7,56
4,webqsp,intermediate_active,ALL,unique_frontier_nodes,312,5.471154,1.0,5.00,16.0,27.7,56



[Hop-wise rho_h]


...


[Question-level work distributions]


,dataset,metric,n,mean,median,p75,p90,p95,max
0,cwq,active_prefixes_total,3519,16.436772,6.0,14.0,36.2,53.0,1403
1,cwq,edges_examined_total,3519,1493.967604,744.0,1793.0,4858.6,5457.0,52146
2,cwq,candidate_branches_total,3519,70.236147,15.0,44.0,112.2,220.3,21666
3,cwq,retrieved_paths_total,3519,58.421427,9.0,30.0,85.0,175.0,21377
4,cwq,peak_frontier_size_question,3519,6.531117,1.0,5.0,16.0,30.0,603



[Reachability]


,dataset,plans,reachable_plans,plan_reachability_pct,questions,reachable_questions,question_reachability_pct
0,cwq,10529,3971,37.714883,3519,2425,68.911623



RQ1 development descriptive summaries saved.


## Suffix-reachability DP (oracle machinery), brute-force validated

In [75]:
# ## 2.7 Suffix-reachability DP -- validation for oracle diagnostics + future training labels
# Gold answers are used OFFLINE only.
# On validation: used only for oracle diagnostics / decision-gate analysis.
# Later on training data: the same reachability machinery may be used to construct AFP supervision labels.
# Gold answers are NEVER available to AFP at inference time.

import random

def suffix_reachable_dp(graph, rule, gold_answers):
    """
    reachable[h][node] = True iff `node` can reach a gold answer
    by following exactly rule[h:].

    h ranges from 0..L.
    h == L is the base case: the current node itself must be a gold answer.
    """
    L = len(rule)
    reachable = [dict() for _ in range(L + 1)]

    for node in graph.nodes():
        reachable[L][node] = node in gold_answers

    for h in range(L - 1, -1, -1):
        rel = rule[h]

        for node in graph.nodes():
            reachable[h][node] = any(
                graph[node][neighbor]["relation"] == rel
                and reachable[h + 1].get(neighbor, False)
                for neighbor in graph.neighbors(node)
            )

    return reachable


def brute_force_suffix_reachable(graph, node, rule_suffix, gold_answers):
    if len(rule_suffix) == 0:
        return node in gold_answers

    if node not in graph:
        return False

    required_rel = rule_suffix[0]

    for neighbor in graph.neighbors(node):
        if graph[node][neighbor]["relation"] != required_rel:
            continue

        if brute_force_suffix_reachable(
            graph,
            neighbor,
            rule_suffix[1:],
            gold_answers
        ):
            return True

    return False


def validate_suffix_dp(
    planning_recs,
    dataset_label,
    n_questions=30,
    n_nodes=20,
    seed=0
):
    rng = random.Random(seed)

    sample_size = min(n_questions, len(planning_recs))
    validation_sample = rng.sample(planning_recs, sample_size)

    dp_checks = 0
    dp_mismatches = 0

    for rec in validation_sample:
        graph = build_graph(rec["graph"])
        gold_answers = set(rec["a_entity"])
        graph_nodes = list(graph.nodes())

        if len(graph_nodes) <= n_nodes:
            nodes_to_check = graph_nodes
        else:
            nodes_to_check = rng.sample(graph_nodes, n_nodes)

        for rule in rec["predicted_paths"]:
            if len(rule) == 0:
                continue

            reachable = suffix_reachable_dp(
                graph,
                rule,
                gold_answers
            )

            for h in range(len(rule) + 1):
                suffix = rule[h:]

                for node in nodes_to_check:
                    dp_checks += 1

                    dp_value = reachable[h].get(node, False)

                    brute_value = brute_force_suffix_reachable(
                        graph,
                        node,
                        suffix,
                        gold_answers
                    )

                    if dp_value != brute_value:
                        dp_mismatches += 1

    print(
        f"[{dataset_label}] DP vs brute-force: "
        f"{dp_checks} (node, hop) states checked | "
        f"mismatches: {dp_mismatches}"
    )

    assert dp_mismatches == 0, (
        f"{dataset_label}: suffix-reachability DP diverges "
        f"from brute force -- fix before oracle diagnostics."
    )

    return dp_checks, dp_mismatches


# =========================================================
# Validate on BOTH development datasets
# =========================================================
webqsp_dp_checks, webqsp_dp_mismatches = validate_suffix_dp(
    webqsp_val_planning,
    "WebQSP validation",
    n_questions=30,
    n_nodes=20,
    seed=0
)

cwq_dp_checks, cwq_dp_mismatches = validate_suffix_dp(
    cwq_val_planning,
    "CWQ validation",
    n_questions=30,
    n_nodes=20,
    seed=1
)

assert webqsp_dp_mismatches == 0
assert cwq_dp_mismatches == 0

print("\n=== SUFFIX-REACHABILITY VALIDATION: PASSED ===")
print(f"WebQSP: {webqsp_dp_checks} states, 0 mismatches")
print(f"CWQ:    {cwq_dp_checks} states, 0 mismatches")
print("DP validated against brute force. Safe to proceed to oracle headroom.")

[WebQSP validation] DP vs brute-force: 4420 (node, hop) states checked | mismatches: 0
[CWQ validation] DP vs brute-force: 4999 (node, hop) states checked | mismatches: 0

=== SUFFIX-REACHABILITY VALIDATION: PASSED ===
WebQSP: 4420 states, 0 mismatches
CWQ:    4999 states, 0 mismatches
DP validated against brute force. Safe to proceed to oracle headroom.


## Validation oracle headroom and development decision gate

In [76]:
# ## 2.8 Oracle headroom on validation + decision gate
# ORACLE DIAGNOSTIC -- NOT AN INFERENCE METHOD.
# Separate read-only re-traversal using validation gold answers offline only.
# Does not modify RoG retrieval, RQ1 profiling, AFP training, or inference-time decisions.
#
# IMPORTANT:
# - Prefix productivity uses reachable_dp[h][current_node].
# - Candidate-branch productivity uses reachable_dp[h+1][neighbor].
# - Only intermediate-hop doomed candidates are oracle-prunable under final-hop protection.
# - Current-hop edge examination is already incurred before AFP acts.
# - Therefore avoidable edge work is downstream work from doomed prefixes at h > 0.

from collections import deque
import numpy as np
import pandas as pd
import os

def oracle_headroom_for_plan(graph, start_node, rule, reachable_dp):
    L = len(rule)

    edges_prod_prefix = [0] * L
    edges_doom_prefix = [0] * L

    branch_prod = [0] * L
    branch_doom = [0] * L

    branch_prunable = [0] * L
    branch_doom_final_protected = [0] * L
    avoidable_downstream_edges = [0] * L

    queue = deque([(start_node, [])])

    while queue:
        current_node, current_path = queue.popleft()
        h = len(current_path)

        if h >= L:
            continue

        if current_node not in graph:
            continue

        # Is this CURRENT PREFIX capable of reaching gold through rule[h:]?
        prefix_is_productive = reachable_dp[h].get(current_node, False)

        for neighbor in graph.neighbors(current_node):
            # Every adjacency entry is examined before relation filtering.
            if prefix_is_productive:
                edges_prod_prefix[h] += 1
            else:
                edges_doom_prefix[h] += 1

                # If h > 0, this doomed prefix was generated at the previous
                # intermediate hop and could theoretically have been removed
                # by a perfect oracle before this expansion occurred.
                if h > 0:
                    avoidable_downstream_edges[h] += 1

            rel = graph[current_node][neighbor]["relation"]

            if rel != rule[h]:
                continue

            # Candidate branch:
            # after taking rule[h], candidate endpoint is evaluated using
            # the remaining suffix rule[h+1:].
            branch_is_productive = reachable_dp[h + 1].get(neighbor, False)

            if branch_is_productive:
                branch_prod[h] += 1
            else:
                branch_doom[h] += 1

                if h < L - 1:
                    # AFP can prune only intermediate candidates.
                    branch_prunable[h] += 1
                else:
                    # Final-hop candidates are protected in the main AFP policy.
                    branch_doom_final_protected[h] += 1

            queue.append(
                (neighbor, current_path + [(current_node, rel, neighbor)])
            )

    return {
        "edges_from_productive_prefix": edges_prod_prefix,
        "edges_from_doomed_prefix": edges_doom_prefix,
        "branches_productive": branch_prod,
        "branches_doomed": branch_doom,
        "oracle_prunable_branches": branch_prunable,
        "doomed_final_branches_protected": branch_doom_final_protected,
        "oracle_avoidable_downstream_edges": avoidable_downstream_edges,
    }


def compute_oracle_headroom(planning_recs, dataset_label):
    rows = []

    for rec in tqdm(
        planning_recs,
        desc=f"Oracle headroom ({dataset_label})"
    ):
        graph = build_graph(rec["graph"])
        gold_answers = set(rec["a_entity"])

        for plan_rank, rule in enumerate(rec["predicted_paths"]):
            L = len(rule)

            if L == 0:
                continue

            reachable_dp = suffix_reachable_dp(
                graph,
                rule,
                gold_answers
            )

            plan_gold_reachable = any(
                reachable_dp[0].get(entity, False)
                for entity in rec["q_entity"]
            )

            for entity in rec["q_entity"]:
                hd = oracle_headroom_for_plan(
                    graph,
                    entity,
                    rule,
                    reachable_dp
                )

                for h in range(L):
                    rows.append({
                        "dataset": dataset_label,
                        "question_id": rec["id"],
                        "plan_id": f"{rec['id']}_p{plan_rank}",
                        "plan_rank": plan_rank,
                        "plan_length": L,
                        "topic_entity": entity,
                        "hop": h,
                        "required_relation": rule[h],
                        "is_intermediate_hop": h < L - 1,
                        "plan_gold_reachable": plan_gold_reachable,

                        "edges_from_productive_prefix":
                            hd["edges_from_productive_prefix"][h],

                        "edges_from_doomed_prefix":
                            hd["edges_from_doomed_prefix"][h],

                        "branches_productive":
                            hd["branches_productive"][h],

                        "branches_doomed":
                            hd["branches_doomed"][h],

                        "oracle_prunable_branches":
                            hd["oracle_prunable_branches"][h],

                        "doomed_final_branches_protected":
                            hd["doomed_final_branches_protected"][h],

                        "oracle_avoidable_downstream_edges":
                            hd["oracle_avoidable_downstream_edges"][h],
                    })

    return pd.DataFrame(rows)


# =========================================================
# Run oracle diagnostic on frozen validation plans
# =========================================================
df_oracle_webqsp = compute_oracle_headroom(
    webqsp_val_planning,
    "webqsp"
)

df_oracle_cwq = compute_oracle_headroom(
    cwq_val_planning,
    "cwq"
)


# =========================================================
# Save raw oracle logs
# =========================================================
df_oracle_webqsp.to_csv(
    os.path.join(
        RQ1_DEV_DIR,
        "oracle_headroom_webqsp_dev.csv"
    ),
    index=False
)

df_oracle_cwq.to_csv(
    os.path.join(
        RQ1_DEV_DIR,
        "oracle_headroom_cwq_dev.csv"
    ),
    index=False
)


# =========================================================
# Consistency check:
# oracle re-traversal must reproduce Cell 141 edge/branch totals
# =========================================================
def check_oracle_vs_rq1(df_oracle, df_rq1, label):
    oracle_edges = (
        df_oracle["edges_from_productive_prefix"].sum()
        + df_oracle["edges_from_doomed_prefix"].sum()
    )

    oracle_branches = (
        df_oracle["branches_productive"].sum()
        + df_oracle["branches_doomed"].sum()
    )

    rq1_edges = df_rq1["edges_examined"].sum()
    rq1_branches = df_rq1["candidate_branches"].sum()

    print(f"\n[{label}] ORACLE vs RQ1 CONSISTENCY")
    print(
        f"Edges: oracle={oracle_edges} | "
        f"RQ1={rq1_edges} | "
        f"match={oracle_edges == rq1_edges}"
    )
    print(
        f"Branches: oracle={oracle_branches} | "
        f"RQ1={rq1_branches} | "
        f"match={oracle_branches == rq1_branches}"
    )

    assert oracle_edges == rq1_edges, (
        f"{label}: oracle edge total does not match RQ1 profiler."
    )

    assert oracle_branches == rq1_branches, (
        f"{label}: oracle branch total does not match RQ1 profiler."
    )


check_oracle_vs_rq1(
    df_oracle_webqsp,
    df_rq1_webqsp_dev,
    "WebQSP validation"
)

check_oracle_vs_rq1(
    df_oracle_cwq,
    df_rq1_cwq_dev,
    "CWQ validation"
)


# =========================================================
# Oracle headroom summary
# =========================================================
def safe_pct(num, den):
    return 100 * num / den if den > 0 else np.nan


def summarize_headroom(df, label):
    total_edges = (
        df["edges_from_productive_prefix"].sum()
        + df["edges_from_doomed_prefix"].sum()
    )

    doomed_prefix_edges = df["edges_from_doomed_prefix"].sum()

    # Edge work theoretically removable because the doomed prefix
    # could have been pruned at the preceding intermediate hop.
    avoidable_edges = df[
        "oracle_avoidable_downstream_edges"
    ].sum()

    total_branches = (
        df["branches_productive"].sum()
        + df["branches_doomed"].sum()
    )

    # Only intermediate candidates are eligible for AFP pruning.
    intermediate_df = df[df["is_intermediate_hop"]]

    intermediate_branches = (
        intermediate_df["branches_productive"].sum()
        + intermediate_df["branches_doomed"].sum()
    )

    prunable_branches = (
        intermediate_df["oracle_prunable_branches"].sum()
    )

    final_protected_doomed = (
        df["doomed_final_branches_protected"].sum()
    )

    print(
        f"\n[{label}] ORACLE DIAGNOSTIC -- "
        f"NOT AN INFERENCE METHOD"
    )

    print(
        f"[{label}] Total examined edges: {total_edges}"
    )

    print(
        f"[{label}] Edges from doomed prefixes: "
        f"{doomed_prefix_edges} "
        f"({safe_pct(doomed_prefix_edges, total_edges):.2f}%)"
    )

    print(
        f"[{label}] Oracle-avoidable downstream edges: "
        f"{avoidable_edges} "
        f"({safe_pct(avoidable_edges, total_edges):.2f}% "
        f"of all examined edges)"
    )

    print(
        f"[{label}] Total relation-valid candidate branches: "
        f"{total_branches}"
    )

    print(
        f"[{label}] Intermediate candidate branches: "
        f"{intermediate_branches}"
    )

    print(
        f"[{label}] Oracle-prunable intermediate branches: "
        f"{prunable_branches} "
        f"({safe_pct(prunable_branches, intermediate_branches):.2f}%)"
    )

    print(
        f"[{label}] Doomed final-hop branches protected by policy: "
        f"{final_protected_doomed}"
    )

    # Conservative view restricted to relation plans that actually
    # contain at least one gold-reaching path.
    reachable_plan_df = df[df["plan_gold_reachable"]]
    reachable_intermediate_df = reachable_plan_df[
        reachable_plan_df["is_intermediate_hop"]
    ]

    reachable_intermediate_branches = (
        reachable_intermediate_df["branches_productive"].sum()
        + reachable_intermediate_df["branches_doomed"].sum()
    )

    reachable_prunable = (
        reachable_intermediate_df["oracle_prunable_branches"].sum()
    )

    reachable_avoidable_edges = (
        reachable_plan_df[
            "oracle_avoidable_downstream_edges"
        ].sum()
    )

    reachable_total_edges = (
        reachable_plan_df["edges_from_productive_prefix"].sum()
        + reachable_plan_df["edges_from_doomed_prefix"].sum()
    )

    print(
        f"[{label}] Gold-reachable plans only -- "
        f"oracle-prunable intermediate branches: "
        f"{reachable_prunable}/"
        f"{reachable_intermediate_branches} "
        f"({safe_pct(reachable_prunable, reachable_intermediate_branches):.2f}%)"
    )

    print(
        f"[{label}] Gold-reachable plans only -- "
        f"oracle-avoidable downstream edges: "
        f"{reachable_avoidable_edges}/"
        f"{reachable_total_edges} "
        f"({safe_pct(reachable_avoidable_edges, reachable_total_edges):.2f}%)"
    )

    # Per-hop summary
    per_hop = (
        df.groupby("hop", as_index=False)
        .agg(
            edges_from_productive_prefix=(
                "edges_from_productive_prefix", "sum"
            ),
            edges_from_doomed_prefix=(
                "edges_from_doomed_prefix", "sum"
            ),
            oracle_avoidable_downstream_edges=(
                "oracle_avoidable_downstream_edges", "sum"
            ),
            branches_productive=(
                "branches_productive", "sum"
            ),
            branches_doomed=(
                "branches_doomed", "sum"
            ),
            oracle_prunable_branches=(
                "oracle_prunable_branches", "sum"
            ),
            doomed_final_branches_protected=(
                "doomed_final_branches_protected", "sum"
            ),
        )
    )

    per_hop["edges_total"] = (
        per_hop["edges_from_productive_prefix"]
        + per_hop["edges_from_doomed_prefix"]
    )

    per_hop["avoidable_edge_pct"] = np.where(
        per_hop["edges_total"] > 0,
        100
        * per_hop["oracle_avoidable_downstream_edges"]
        / per_hop["edges_total"],
        np.nan
    )

    per_hop["candidate_branches_total"] = (
        per_hop["branches_productive"]
        + per_hop["branches_doomed"]
    )

    print(f"\n[{label}] Per-hop oracle headroom")
    display(per_hop)

    summary = pd.DataFrame([{
        "dataset": label,
        "total_edges": total_edges,
        "doomed_prefix_edges": doomed_prefix_edges,
        "oracle_avoidable_downstream_edges": avoidable_edges,
        "oracle_avoidable_edge_pct":
            safe_pct(avoidable_edges, total_edges),

        "total_candidate_branches": total_branches,
        "intermediate_candidate_branches": intermediate_branches,
        "oracle_prunable_intermediate_branches": prunable_branches,
        "oracle_prunable_branch_pct":
            safe_pct(prunable_branches, intermediate_branches),

        "doomed_final_branches_protected":
            final_protected_doomed,

        "reachable_plan_intermediate_branches":
            reachable_intermediate_branches,

        "reachable_plan_oracle_prunable_branches":
            reachable_prunable,

        "reachable_plan_prunable_branch_pct":
            safe_pct(
                reachable_prunable,
                reachable_intermediate_branches
            ),

        "reachable_plan_total_edges":
            reachable_total_edges,

        "reachable_plan_oracle_avoidable_edges":
            reachable_avoidable_edges,

        "reachable_plan_avoidable_edge_pct":
            safe_pct(
                reachable_avoidable_edges,
                reachable_total_edges
            ),
    }])

    return summary, per_hop


webqsp_headroom_summary, webqsp_headroom_hop = (
    summarize_headroom(
        df_oracle_webqsp,
        "WebQSP validation"
    )
)

cwq_headroom_summary, cwq_headroom_hop = (
    summarize_headroom(
        df_oracle_cwq,
        "CWQ validation"
    )
)


# =========================================================
# Save oracle summaries
# =========================================================
webqsp_headroom_summary.to_csv(
    os.path.join(
        RQ1_DEV_DIR,
        "oracle_headroom_webqsp_dev_summary.csv"
    ),
    index=False
)

cwq_headroom_summary.to_csv(
    os.path.join(
        RQ1_DEV_DIR,
        "oracle_headroom_cwq_dev_summary.csv"
    ),
    index=False
)

webqsp_headroom_hop.to_csv(
    os.path.join(
        RQ1_DEV_DIR,
        "oracle_headroom_webqsp_dev_hop.csv"
    ),
    index=False
)

cwq_headroom_hop.to_csv(
    os.path.join(
        RQ1_DEV_DIR,
        "oracle_headroom_cwq_dev_hop.csv"
    ),
    index=False
)


# =========================================================
# Development decision gate -- evidence-based answers
# =========================================================

def get_summary_value(df, col):
    return float(df.iloc[0][col])

# RQ1 decision-opportunity evidence
w_opp = rq1_webqsp_summary["opportunity"].iloc[0]
c_opp = rq1_cwq_summary["opportunity"].iloc[0]

w_decision_hop_pct = float(w_opp["decision_hop_pct"])
c_decision_hop_pct = float(c_opp["decision_hop_pct"])

w_question_decision_pct = float(w_opp["questions_with_decision_pct"])
c_question_decision_pct = float(c_opp["questions_with_decision_pct"])

w_eligible_decision_pct = float(w_opp["decision_pct_among_eligible_questions"])
c_eligible_decision_pct = float(c_opp["decision_pct_among_eligible_questions"])

# Oracle evidence
w_avoidable_edge_pct = get_summary_value(
    webqsp_headroom_summary, "oracle_avoidable_edge_pct"
)
c_avoidable_edge_pct = get_summary_value(
    cwq_headroom_summary, "oracle_avoidable_edge_pct"
)

w_prunable_branch_pct = get_summary_value(
    webqsp_headroom_summary, "oracle_prunable_branch_pct"
)
c_prunable_branch_pct = get_summary_value(
    cwq_headroom_summary, "oracle_prunable_branch_pct"
)

w_reachable_prunable_pct = get_summary_value(
    webqsp_headroom_summary, "reachable_plan_prunable_branch_pct"
)
c_reachable_prunable_pct = get_summary_value(
    cwq_headroom_summary, "reachable_plan_prunable_branch_pct"
)

w_reachable_avoidable_pct = get_summary_value(
    webqsp_headroom_summary, "reachable_plan_avoidable_edge_pct"
)
c_reachable_avoidable_pct = get_summary_value(
    cwq_headroom_summary, "reachable_plan_avoidable_edge_pct"
)

w_total_edges = int(
    webqsp_headroom_summary.iloc[0]["total_edges"]
)
c_total_edges = int(
    cwq_headroom_summary.iloc[0]["total_edges"]
)

w_total_branches = int(
    webqsp_headroom_summary.iloc[0]["total_candidate_branches"]
)
c_total_branches = int(
    cwq_headroom_summary.iloc[0]["total_candidate_branches"]
)

w_edges_per_branch = w_total_edges / w_total_branches
c_edges_per_branch = c_total_edges / c_total_branches

# Duplicate-pressure evidence from Cell 143
w_dup = rq1_webqsp_summary["duplicate"]
c_dup = rq1_cwq_summary["duplicate"]

w_prefix_dup_pct = float(
    w_dup[
        (w_dup["hop"] == "ALL") &
        (w_dup["metric"] == "prefix_to_unique_expanded_ratio")
    ]["ratio_gt_1_pct"].iloc[0]
)

c_prefix_dup_pct = float(
    c_dup[
        (c_dup["hop"] == "ALL") &
        (c_dup["metric"] == "prefix_to_unique_expanded_ratio")
    ]["ratio_gt_1_pct"].iloc[0]
)

w_frontier_dup_pct = float(
    w_dup[
        (w_dup["hop"] == "ALL") &
        (w_dup["metric"] == "candidate_to_unique_frontier_ratio")
    ]["ratio_gt_1_pct"].iloc[0]
)

c_frontier_dup_pct = float(
    c_dup[
        (c_dup["hop"] == "ALL") &
        (c_dup["metric"] == "candidate_to_unique_frontier_ratio")
    ]["ratio_gt_1_pct"].iloc[0]
)

# Hop contributing the largest total oracle-avoidable edge work
w_max_row = webqsp_headroom_hop.loc[
    webqsp_headroom_hop["oracle_avoidable_downstream_edges"].idxmax()
]

c_max_row = cwq_headroom_hop.loc[
    cwq_headroom_hop["oracle_avoidable_downstream_edges"].idxmax()
]

w_max_hop = int(w_max_row["hop"])
c_max_hop = int(c_max_row["hop"])

w_max_hop_edges = int(
    w_max_row["oracle_avoidable_downstream_edges"]
)
c_max_hop_edges = int(
    c_max_row["oracle_avoidable_downstream_edges"]
)


print("\n" + "=" * 80)
print("DEVELOPMENT DECISION GATE -- VALIDATION EVIDENCE")
print("=" * 80)

print("\nQ1. Is residual relation-valid branching frequent enough to justify selective pruning?")
print(
    f"ANSWER: YES. Actual intermediate decision opportunities occur in "
    f"{w_decision_hop_pct:.2f}% of WebQSP and "
    f"{c_decision_hop_pct:.2f}% of CWQ intermediate active hops. "
    f"At question level, {w_question_decision_pct:.2f}% of WebQSP and "
    f"{c_question_decision_pct:.2f}% of CWQ questions contain at least one "
    f"real AFP decision opportunity."
)

print("\nQ2. At which hops and in which dataset is downstream pruning opportunity largest?")
print(
    f"ANSWER: CWQ provides substantially larger downstream structural headroom. "
    f"Oracle-avoidable downstream edges represent {c_avoidable_edge_pct:.2f}% "
    f"of all CWQ examined edges versus {w_avoidable_edge_pct:.2f}% for WebQSP. "
    f"By total oracle-avoidable edge work, WebQSP hop {w_max_hop} contributes "
    f"{w_max_hop_edges:,} edges, while CWQ hop {c_max_hop} contributes "
    f"{c_max_hop_edges:,} edges."
)

print("\nQ3. Is candidate-branch count or edge examination the more appropriate graph-search cost?")
print(
    f"ANSWER: EXAMINED EDGES should remain the primary structural-cost metric. "
    f"WebQSP performs {w_total_edges:,} adjacency examinations for "
    f"{w_total_branches:,} relation-valid branches "
    f"(~{w_edges_per_branch:.2f} examined edges per branch), while CWQ performs "
    f"{c_total_edges:,} examinations for {c_total_branches:,} branches "
    f"(~{c_edges_per_branch:.2f} per branch). Candidate count alone therefore "
    f"does not capture the graph work required before relation matching."
)

print("\nQ4. Are duplicate path/frontier states the main explanation for residual branching?")
print(
    f"ANSWER: NO. Repeated-prefix expansion is rare "
    f"({w_prefix_dup_pct:.2f}% WebQSP; {c_prefix_dup_pct:.2f}% CWQ), and "
    f"candidate-endpoint duplication is also limited "
    f"({w_frontier_dup_pct:.2f}% WebQSP; {c_frontier_dup_pct:.2f}% CWQ). "
    f"Residual search growth is therefore mainly genuine relation-valid "
    f"branch multiplicity rather than simple duplication."
)

print("\nQ5. How do pruning-opportunity frequency and oracle headroom differ between datasets?")
print(
    f"ANSWER: WebQSP has more frequent intermediate decision opportunities "
    f"({w_decision_hop_pct:.2f}% vs {c_decision_hop_pct:.2f}% of active "
    f"intermediate hops), but CWQ has substantially greater downstream edge-saving "
    f"headroom ({c_avoidable_edge_pct:.2f}% vs {w_avoidable_edge_pct:.2f}%). "
    f"Thus CWQ is not necessarily more frequently branching, but its deeper "
    f"retrieval produces greater downstream cost when branching occurs."
)

print("\nQ6. Within gold-reachable plans, does substantial oracle-prunable branching remain?")
print(
    f"ANSWER: YES. Within gold-reachable plans, "
    f"{w_reachable_prunable_pct:.2f}% of WebQSP and "
    f"{c_reachable_prunable_pct:.2f}% of CWQ intermediate candidate branches "
    f"are oracle-doomed. Their corresponding oracle-avoidable examined-edge "
    f"headroom is {w_reachable_avoidable_pct:.2f}% and "
    f"{c_reachable_avoidable_pct:.2f}%, respectively. "
    f"This confirms that pruning opportunity is not explained only by plans "
    f"that already fail to reach a gold answer."
)

print("\nQ7. Is the structural headroom sufficient to justify AFP development?")
print(
    f"ANSWER: YES. Residual branching is non-trivial, real branch-selection "
    f"opportunities exist in both datasets, and oracle analysis identifies "
    f"{w_prunable_branch_pct:.2f}% of WebQSP and "
    f"{c_prunable_branch_pct:.2f}% of CWQ intermediate candidate branches as "
    f"theoretically removable. The evidence therefore justifies proceeding "
    f"to AFP development and controlled RQ2 evaluation."
)

print("\n" + "=" * 80)
print("=== DEVELOPMENT DECISION GATE: PASSED ===")
print("=" * 80)

print(
    "Conclusion: Development RQ1 establishes a meaningful residual search-space "
    "problem and sufficient oracle pruning headroom to proceed to AFP/RQ2."
)

print(
    "Caution: These results establish structural opportunity only. Actual search "
    "reduction, answer preservation, runtime benefit, and pruning overhead remain "
    "unverified until RQ2/RQ3 experiments are completed."
)

print(
    "Protocol: All conclusions above use frozen validation evidence only; "
    "no frozen-test results are used for AFP development decisions."
)

Oracle headroom (webqsp):   0%|          | 0/246 [00:00<?, ?it/s]

Oracle headroom (cwq):   0%|          | 0/3519 [00:00<?, ?it/s]


[WebQSP validation] ORACLE vs RQ1 CONSISTENCY
Edges: oracle=341526 | RQ1=341526 | match=True
Branches: oracle=7983 | RQ1=7983 | match=True

[CWQ validation] ORACLE vs RQ1 CONSISTENCY
Edges: oracle=5257272 | RQ1=5257272 | match=True
Branches: oracle=247161 | RQ1=247161 | match=True

[WebQSP validation] ORACLE DIAGNOSTIC -- NOT AN INFERENCE METHOD
[WebQSP validation] Total examined edges: 341526
[WebQSP validation] Edges from doomed prefixes: 159039 (46.57%)
[WebQSP validation] Oracle-avoidable downstream edges: 11717 (3.43% of all examined edges)
[WebQSP validation] Total relation-valid candidate branches: 7983
[WebQSP validation] Intermediate candidate branches: 1707
[WebQSP validation] Oracle-prunable intermediate branches: 1340 (78.50%)
[WebQSP validation] Doomed final-hop branches protected by policy: 4882
[WebQSP validation] Gold-reachable plans only -- oracle-prunable intermediate branches: 647/1014 (63.81%)
[WebQSP validation] Gold-reachable plans only -- oracle-avoidable downst

,hop,edges_from_productive_prefix,edges_from_doomed_prefix,oracle_avoidable_downstream_edges,branches_productive,branches_doomed,oracle_prunable_branches,doomed_final_branches_protected,edges_total,avoidable_edge_pct,candidate_branches_total
0,0,173107,147322,0,1144,2554,1340,1214,320429,0.000000,3698
1,1,9380,11717,11717,617,3668,0,3668,21097,55.538702,4285



[CWQ validation] ORACLE DIAGNOSTIC -- NOT AN INFERENCE METHOD
[CWQ validation] Total examined edges: 5257272
[CWQ validation] Edges from doomed prefixes: 3197488 (60.82%)
[CWQ validation] Oracle-avoidable downstream edges: 1176294 (22.37% of all examined edges)
[CWQ validation] Total relation-valid candidate branches: 247161
[CWQ validation] Intermediate candidate branches: 41576
[CWQ validation] Oracle-prunable intermediate branches: 34431 (82.81%)
[CWQ validation] Doomed final-hop branches protected by policy: 195499
[CWQ validation] Gold-reachable plans only -- oracle-prunable intermediate branches: 14008/21153 (66.22%)
[CWQ validation] Gold-reachable plans only -- oracle-avoidable downstream edges: 196144/2728920 (7.19%)

[CWQ validation] Per-hop oracle headroom


,hop,edges_from_productive_prefix,edges_from_doomed_prefix,oracle_avoidable_downstream_edges,branches_productive,branches_doomed,oracle_prunable_branches,doomed_final_branches_protected,edges_total,avoidable_edge_pct,candidate_branches_total
0,0,1186111,2021194,0,6018,52867,28511,24356,3207305,0.000000,58885
1,1,666705,1036547,1036547,6489,81488,2679,78809,1703252,60.856937,87977
2,2,55475,45373,45373,1901,39543,3230,36313,100848,44.991472,41444
3,3,107617,94287,94287,1351,51999,11,51988,201904,46.698926,53350
4,4,43876,87,87,1472,4033,0,4033,43963,0.197894,5505



DEVELOPMENT DECISION GATE -- VALIDATION EVIDENCE

Q1. Is residual relation-valid branching frequent enough to justify selective pruning?
ANSWER: YES. Actual intermediate decision opportunities occur in 47.12% of WebQSP and 33.77% of CWQ intermediate active hops. At question level, 31.30% of WebQSP and 39.93% of CWQ questions contain at least one real AFP decision opportunity.

Q2. At which hops and in which dataset is downstream pruning opportunity largest?
ANSWER: CWQ provides substantially larger downstream structural headroom. Oracle-avoidable downstream edges represent 22.37% of all CWQ examined edges versus 3.43% for WebQSP. By total oracle-avoidable edge work, WebQSP hop 1 contributes 11,717 edges, while CWQ hop 1 contributes 1,036,547 edges.

Q3. Is candidate-branch count or edge examination the more appropriate graph-search cost?
ANSWER: EXAMINED EDGES should remain the primary structural-cost metric. WebQSP performs 341,526 adjacency examinations for 7,983 relation-valid bran

## Search-space characterization of unpruned RoG. Values summarize the observed hop-level distributions.

In [77]:
# ## RQ1 supervisor table -- development validation results
# Table reports distributions over ALL ACTIVE question x plan x hop frontiers.
# These are development/validation results, NOT final frozen-test RQ1 results.

import os
import pandas as pd
from IPython.display import display

METRICS = {
    "active_prefixes": "Active Path Prefixes",
    "unique_expanded_nodes": "Unique Expanded Nodes",
    "edges_examined": "Edges Examined",
    "candidate_branches": "Candidate Branches",
    "unique_frontier_nodes": "Unique Frontier Nodes",
}

def make_rq1_supervisor_table(df, dataset_name):
    active = df[df["active_prefixes"] > 0].copy()
    rows = []

    for col, label in METRICS.items():
        s = active[col].dropna()

        rows.append({
            "Dataset": dataset_name,
            "Metric": label,
            "Median": s.median(),
            "P90": s.quantile(0.90),
            "P95": s.quantile(0.95),
            "Maximum": s.max(),
        })

    return pd.DataFrame(rows)

table_webqsp = make_rq1_supervisor_table(
    df_rq1_webqsp_dev,
    "WebQSP"
)

table_cwq = make_rq1_supervisor_table(
    df_rq1_cwq_dev,
    "CWQ"
)

rq1_supervisor_table = pd.concat(
    [table_webqsp, table_cwq],
    ignore_index=True
)

# Clean display formatting
for col in ["Median", "P90", "P95", "Maximum"]:
    rq1_supervisor_table[col] = rq1_supervisor_table[col].round(1)

display(rq1_supervisor_table)

# Save for paper drafting
rq1_supervisor_table.to_csv(
    os.path.join(
        RQ1_DEV_DIR,
        "rq1_development_search_space_summary.csv"
    ),
    index=False
)

print("\nDevelopment RQ1 search-space table saved.")
print("NOTE: Values are from frozen VALIDATION plans, not final test reporting.")

,Dataset,Metric,Median,P90,P95,Maximum
0,WebQSP,Active Path Prefixes,1.0,4.0,9.5,56
1,WebQSP,Unique Expanded Nodes,1.0,4.0,9.5,56
2,WebQSP,Edges Examined,135.0,1475.0,1686.0,1969
3,WebQSP,Candidate Branches,1.0,18.0,35.0,533
4,WebQSP,Unique Frontier Nodes,1.0,17.0,33.0,145
5,CWQ,Active Path Prefixes,1.0,5.0,13.0,603
6,CWQ,Unique Expanded Nodes,1.0,5.0,12.0,210
7,CWQ,Edges Examined,110.0,1111.7,1691.0,23290
8,CWQ,Candidate Branches,1.0,21.0,37.0,21372
9,CWQ,Unique Frontier Nodes,1.0,19.0,33.0,333



Development RQ1 search-space table saved.
NOTE: Values are from frozen VALIDATION plans, not final test reporting.
